In [1]:
import pandas as pd
import numpy as np
import json
import re
from ast import literal_eval
import importlib
from dotenv import load_dotenv
load_dotenv()
import os
os.chdir(os.getenv('PARENT_DIR'))

In [2]:
from glob import glob
import shutil

In [3]:
model_paths = glob('outputs_seq2seq/models/hotel_reviews/*/mvp_aos/*/*')
eval_paths = glob('outputs_seq2seq/evals/hotel_reviews/*/mvp_aos/*')
id2seedlang_model = {model_path.split('/')[-1]: (model_path.split('/')[-2], model_path.split('/')[-4]) for model_path in model_paths}

In [4]:
model_paths

['outputs_seq2seq/models/hotel_reviews/jav/mvp_aos/seed_123/20260408_105104_train_model-mt5-base_lr-0.0002_bs-16_epochs-20',
 'outputs_seq2seq/models/hotel_reviews/jav/mvp_aos/seed_31415/20260408_110353_train_model-mt5-base_lr-0.0002_bs-16_epochs-20',
 'outputs_seq2seq/models/hotel_reviews/jav/mvp_aos/seed_9584/20260408_104439_train_model-mt5-base_lr-0.0002_bs-16_epochs-20',
 'outputs_seq2seq/models/hotel_reviews/jav/mvp_aos/seed_777/20260408_111021_train_model-mt5-base_lr-0.0002_bs-16_epochs-20',
 'outputs_seq2seq/models/hotel_reviews/jav/mvp_aos/seed_2024/20260408_105726_train_model-mt5-base_lr-0.0002_bs-16_epochs-20',
 'outputs_seq2seq/models/hotel_reviews/eng/mvp_aos/seed_123/20260408_102014_train_model-mt5-base_lr-0.0002_bs-16_epochs-20',
 'outputs_seq2seq/models/hotel_reviews/eng/mvp_aos/seed_31415/20260408_103222_train_model-mt5-base_lr-0.0002_bs-16_epochs-20',
 'outputs_seq2seq/models/hotel_reviews/eng/mvp_aos/seed_9584/20260408_101405_train_model-mt5-base_lr-0.0002_bs-16_epoch

In [5]:
id2seedlang_model

{'20260408_105104_train_model-mt5-base_lr-0.0002_bs-16_epochs-20': ('seed_123',
  'jav'),
 '20260408_110353_train_model-mt5-base_lr-0.0002_bs-16_epochs-20': ('seed_31415',
  'jav'),
 '20260408_104439_train_model-mt5-base_lr-0.0002_bs-16_epochs-20': ('seed_9584',
  'jav'),
 '20260408_111021_train_model-mt5-base_lr-0.0002_bs-16_epochs-20': ('seed_777',
  'jav'),
 '20260408_105726_train_model-mt5-base_lr-0.0002_bs-16_epochs-20': ('seed_2024',
  'jav'),
 '20260408_102014_train_model-mt5-base_lr-0.0002_bs-16_epochs-20': ('seed_123',
  'eng'),
 '20260408_103222_train_model-mt5-base_lr-0.0002_bs-16_epochs-20': ('seed_31415',
  'eng'),
 '20260408_101405_train_model-mt5-base_lr-0.0002_bs-16_epochs-20': ('seed_9584',
  'eng'),
 '20260408_103825_train_model-mt5-base_lr-0.0002_bs-16_epochs-20': ('seed_777',
  'eng'),
 '20260408_102618_train_model-mt5-base_lr-0.0002_bs-16_epochs-20': ('seed_2024',
  'eng'),
 '20260408_115323_train_model-mt5-base_lr-0.0002_bs-16_epochs-20': ('seed_123',
  'min'),
 '

In [7]:
for key, (seed, lang) in id2seedlang_model.items():
	# Make seed folder in eval path
	new_eval_path = f'outputs_seq2seq/evals/hotel_reviews/{lang}/mvp_aos/{seed}/{key}'
	os.makedirs(new_eval_path, exist_ok=True)
	# Copy eval dir to new path
	eval_dir = f'outputs_seq2seq/evals/hotel_reviews/{lang}/mvp_aos/{key}'
	shutil.copytree(eval_dir, new_eval_path, dirs_exist_ok=True)
	print(f'Copied {eval_dir} to {new_eval_path}')

Copied outputs_seq2seq/evals/hotel_reviews/jav/mvp_aos/20260408_105104_train_model-mt5-base_lr-0.0002_bs-16_epochs-20 to outputs_seq2seq/evals/hotel_reviews/jav/mvp_aos/seed_123/20260408_105104_train_model-mt5-base_lr-0.0002_bs-16_epochs-20
Copied outputs_seq2seq/evals/hotel_reviews/jav/mvp_aos/20260408_110353_train_model-mt5-base_lr-0.0002_bs-16_epochs-20 to outputs_seq2seq/evals/hotel_reviews/jav/mvp_aos/seed_31415/20260408_110353_train_model-mt5-base_lr-0.0002_bs-16_epochs-20
Copied outputs_seq2seq/evals/hotel_reviews/jav/mvp_aos/20260408_104439_train_model-mt5-base_lr-0.0002_bs-16_epochs-20 to outputs_seq2seq/evals/hotel_reviews/jav/mvp_aos/seed_9584/20260408_104439_train_model-mt5-base_lr-0.0002_bs-16_epochs-20
Copied outputs_seq2seq/evals/hotel_reviews/jav/mvp_aos/20260408_111021_train_model-mt5-base_lr-0.0002_bs-16_epochs-20 to outputs_seq2seq/evals/hotel_reviews/jav/mvp_aos/seed_777/20260408_111021_train_model-mt5-base_lr-0.0002_bs-16_epochs-20
Copied outputs_seq2seq/evals/hote

In [ ]:
# outputs/evals/hotel_reviews/indo/mvp_aos/seed_2024/20260226_063347_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-1560/checkpoint-1560/unconstrained_decoding/exact_match.json
# outputs_seq2seq/evals/hotel_reviews/eng/mvp_aos/seed_31415/20260408_103222_train_model-mt5-base_lr-0.0002_bs-16_epochs-20/checkpoint-3120/unconstrained_decoding/exact_match.json

In [8]:
from glob import glob
from pathlib import Path
import shutil
import filecmp

BASE_DIR = Path("outputs/evals")
DUPLICATED_CHECKPOINT_PATTERN = "outputs/evals/*/*/*/seed_*/*/checkpoint-*/checkpoint-*"
DRY_RUN = False


def is_duplicated_checkpoint_dir(path: Path) -> bool:
    return path.name.startswith("checkpoint-") and path.parent.name == path.name


def move_child(child: Path, target: Path) -> tuple[int, int, int]:
    moved = 0
    skipped = 0
    conflicts = 0

    if child.is_dir():
        if target.exists():
            shutil.copytree(child, target, dirs_exist_ok=True)
            shutil.rmtree(child)
        else:
            shutil.move(str(child), str(target))
        moved += 1
    else:
        if target.exists():
            if filecmp.cmp(str(child), str(target), shallow=False):
                child.unlink()
                skipped += 1
            else:
                conflict_target = target.with_name(f"{target.stem}_from_duplicated_checkpoint{target.suffix}")
                shutil.move(str(child), str(conflict_target))
                conflicts += 1
        else:
            shutil.move(str(child), str(target))
            moved += 1

    return moved, skipped, conflicts


def cleanup_empty_dirs(path: Path, stop_at: Path):
    current = path
    while current != stop_at and current.exists():
        try:
            current.rmdir()
        except OSError:
            break
        current = current.parent


duplicated_dirs = sorted(
    Path(p) for p in glob(DUPLICATED_CHECKPOINT_PATTERN)
    if is_duplicated_checkpoint_dir(Path(p))
)

print(f"Found {len(duplicated_dirs)} duplicated checkpoint directories")

total_moved = 0
total_skipped = 0
total_conflicts = 0

for dup_dir in duplicated_dirs:
    canonical_dir = dup_dir.parent
    print(f"Fixing: {dup_dir} -> {canonical_dir}")

    if DRY_RUN:
        continue

    for child in list(dup_dir.iterdir()):
        target = canonical_dir / child.name
        moved, skipped, conflicts = move_child(child, target)
        total_moved += moved
        total_skipped += skipped
        total_conflicts += conflicts

    cleanup_empty_dirs(dup_dir, BASE_DIR)

remaining = [
    p for p in glob(DUPLICATED_CHECKPOINT_PATTERN)
    if is_duplicated_checkpoint_dir(Path(p))
]

print("Done")
print(f"Moved entries: {total_moved}")
print(f"Skipped identical files: {total_skipped}")
print(f"Conflicts renamed: {total_conflicts}")
print(f"Remaining duplicated checkpoint dirs: {len(remaining)}")

Found 60 duplicated checkpoint directories
Fixing: outputs/evals/hotel_reviews/eng/mvp/seed_123/20260402_091332_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-7760/checkpoint-7760 -> outputs/evals/hotel_reviews/eng/mvp/seed_123/20260402_091332_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-7760
Fixing: outputs/evals/hotel_reviews/eng/mvp/seed_2024/20260402_100706_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-7760/checkpoint-7760 -> outputs/evals/hotel_reviews/eng/mvp/seed_2024/20260402_100706_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-7760
Fixing: outputs/evals/hotel_reviews/eng/mvp/seed_31415/20260402_110041_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-7760/checkpoint-7760 -> outputs/evals/hotel_reviews/eng/mvp/seed_31415/20260402_110041_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs-10/checkpoint-7760
Fixing: outputs/evals/hotel_reviews/eng/mvp/seed_777/20260402_115352_train_model-Qwen2.5-0.5B_lr-5e-05_bs-4_epochs

In [9]:
from glob import glob
from pathlib import Path
import json

BASE_GLOB_PATTERNS = [
    "outputs_seq2seq/evals/**/seed_*/**/exact_match.json",
    "outputs_seq2seq/evals/**/seed_*/**/instruct_absa.json",
    "outputs_seq2seq/evals/**/seed_*/**/semantic_metrics.json",
    "outputs_seq2seq/evals/**/seed_*/**/voting_exact_match.json",
    "outputs_seq2seq/evals/**/seed_*/**/voting_instruct_absa.json",
    "outputs_seq2seq/evals/**/seed_*/**/voting_semantic_matrics.json",
]
DRY_RUN = False


def extract_seed_from_path(path: Path) -> str | None:
    for part in path.parts:
        if part.startswith("seed_") and len(part) > len("seed_"):
            return part[len("seed_"):]
    return None


metric_paths = sorted(
    {p for pattern in BASE_GLOB_PATTERNS for p in glob(pattern, recursive=True)}
)

print(f"Found {len(metric_paths)} metric file(s) to inspect")

updated = 0
unchanged = 0
skipped = 0
errors = 0

for metric_path_str in metric_paths:
    metric_path = Path(metric_path_str)
    seed_from_path = extract_seed_from_path(metric_path)

    if not seed_from_path:
        skipped += 1
        print(f"[SKIP] No seed segment in path: {metric_path}")
        continue

    try:
        with metric_path.open("r", encoding="utf-8") as f:
            data = json.load(f)

        if not isinstance(data, dict):
            skipped += 1
            print(f"[SKIP] Non-dict JSON: {metric_path}")
            continue

        old_seed = data.get("seed")
        new_seed = str(seed_from_path)

        if old_seed == new_seed:
            unchanged += 1
            continue

        data["seed"] = new_seed

        if not DRY_RUN:
            with metric_path.open("w", encoding="utf-8") as f:
                json.dump(data, f, indent=2, ensure_ascii=False)
                f.write("\n")

        updated += 1
        print(f"[FIX] {metric_path} | seed: {old_seed} -> {new_seed}")

    except Exception as exc:
        errors += 1
        print(f"[ERROR] {metric_path}: {exc}")

print("Done")
print(f"Updated: {updated}")
print(f"Unchanged: {unchanged}")
print(f"Skipped: {skipped}")
print(f"Errors: {errors}")

Found 60 metric file(s) to inspect
[FIX] outputs_seq2seq/evals/hotel_reviews/eng/mvp_aos/seed_123/20260408_102014_train_model-mt5-base_lr-0.0002_bs-16_epochs-20/checkpoint-3120/unconstrained_decoding/exact_match.json | seed: None -> 123
[FIX] outputs_seq2seq/evals/hotel_reviews/eng/mvp_aos/seed_123/20260408_102014_train_model-mt5-base_lr-0.0002_bs-16_epochs-20/checkpoint-3120/unconstrained_decoding/instruct_absa.json | seed: None -> 123
[FIX] outputs_seq2seq/evals/hotel_reviews/eng/mvp_aos/seed_2024/20260408_102618_train_model-mt5-base_lr-0.0002_bs-16_epochs-20/checkpoint-3120/unconstrained_decoding/exact_match.json | seed: None -> 2024
[FIX] outputs_seq2seq/evals/hotel_reviews/eng/mvp_aos/seed_2024/20260408_102618_train_model-mt5-base_lr-0.0002_bs-16_epochs-20/checkpoint-3120/unconstrained_decoding/instruct_absa.json | seed: None -> 2024
[FIX] outputs_seq2seq/evals/hotel_reviews/eng/mvp_aos/seed_31415/20260408_103222_train_model-mt5-base_lr-0.0002_bs-16_epochs-20/checkpoint-3120/uncon